In [2]:
# ========================================================
# SIMPLE HANDWRITING AUTO-COMPLETE
# TrOCR (OCR) + GPT-2 (Text Completion)
# ========================================================

import torch
from PIL import Image
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    GPT2LMHeadModel,
    GPT2Tokenizer
)

# -------------------------------
# 1. SETUP
# -------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Device: {device}\n")

# -------------------------------
# 2. LOAD MODELS
# -------------------------------
print("📖 Loading TrOCR (OCR model)...")
ocr_processor = TrOCRProcessor.from_pretrained("microsoft/trocr-large-handwritten")
ocr_model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-large-handwritten"
).to(device)
ocr_model.eval()

print("🤖 Loading GPT-2 (Language model)...")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2-medium")
gpt2_model = GPT2LMHeadModel.from_pretrained(
    "gpt2-medium",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
gpt2_model.eval()

# Set pad token
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

print("✅ Models loaded!\n")


# -------------------------------
# 3. CORE FUNCTIONS
# -------------------------------
def ocr_image(image_path: str) -> str:
    """
    Extract text from handwriting image using TrOCR
    
    Args:
        image_path: Path to the handwriting image
    
    Returns:
        Extracted text string
    """
    print(f"🔍 Reading image: {image_path}")
    
    # Load and preprocess image
    image = Image.open(image_path).convert("RGB")
    pixel_values = ocr_processor(image, return_tensors="pt").pixel_values.to(device)
    
    # Generate text
    with torch.no_grad():
        generated_ids = ocr_model.generate(
            pixel_values,
            max_new_tokens=512,
            do_sample=False  # Use greedy decoding for accuracy
        )
    
    # Decode to text
    text = ocr_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    print(f"📝 Extracted text: '{text}'")
    return text.strip()


def complete_text(text: str, num_words: int = 3) -> str:
    """
    Complete the text using GPT-2
    
    Args:
        text: Input text to complete
        num_words: Number of words to add (default: 3)
    
    Returns:
        Completed text string
    """
    print(f"🤖 Generating {num_words}-word continuation...")
    
    # Tokenize input
    inputs = gpt2_tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)
    
    # Generate continuation
    with torch.no_grad():
        outputs = gpt2_model.generate(
            inputs.input_ids,
            max_new_tokens=num_words * 3,  # More tokens to filter from
            min_new_tokens=2,               # Ensure some output
            temperature=0.7,                # Lower = more focused
            top_p=0.9,                      # Nucleus sampling
            top_k=40,                       # Top-k sampling
            do_sample=True,                 # Enable sampling
            pad_token_id=gpt2_tokenizer.eos_token_id,
            no_repeat_ngram_size=3,         # Avoid repetition
            repetition_penalty=1.2,         # Discourage repetition
        )
    
    # Decode generated text
    generated_text = gpt2_tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the NEW part
    if generated_text.startswith(text):
        continuation = generated_text[len(text):].strip()
    else:
        continuation = generated_text.strip()
    
    # Clean up continuation - remove punctuation that starts new sentences
    # Remove leading periods, commas, etc.
    continuation = continuation.lstrip('.,!?;:')
    
    # Stop at sentence-ending punctuation to avoid new sentences
    for i, char in enumerate(continuation):
        if char in '.!?' and i > 0:
            continuation = continuation[:i]
            break
    
    # Limit to num_words
    words = continuation.split()[:num_words]
    continuation = ' '.join(words)
    
    # Remove any remaining punctuation at the end that might start new thought
    continuation = continuation.rstrip('.,!?;:')
    
    # Combine
    if continuation:
        completed = f"{text} {continuation}"
    else:
        completed = text
    
    print(f"✨ Completed text: '{completed}'")
    
    return completed


# -------------------------------
# 4. MAIN PIPELINE
# -------------------------------
def autocomplete_handwriting(image_path: str, num_words: int = 3):
    """
    Complete pipeline: OCR + Text Completion
    
    Args:
        image_path: Path to handwriting image
        num_words: Number of words to add (default: 3)
    
    Returns:
        Tuple of (original_text, completed_text)
    """
    print("=" * 80)
    print("🚀 HANDWRITING AUTO-COMPLETE")
    print("=" * 80)
    print()
    
    # Step 1: OCR
    print("📋 STEP 1: Optical Character Recognition (OCR)")
    print("-" * 80)
    original_text = ocr_image(image_path)
    print()
    
    # Step 2: Text Completion
    print("📋 STEP 2: Text Completion")
    print("-" * 80)
    completed_text = complete_text(original_text, num_words)
    print()
    
    # Step 3: Results
    print("📋 STEP 3: Results")
    print("-" * 80)
    
    # Show what was added
    original_words = original_text.split()
    completed_words = completed_text.split()
    
    if len(completed_words) > len(original_words):
        added_words = ' '.join(completed_words[len(original_words):])
        print(f"📝 Original:  {original_text}")
        print(f"➕ Added:     {added_words}")
        print(f"✅ Completed: {completed_text}")
    else:
        print(f"📝 Text: {completed_text}")
        print("⚠️ No continuation generated")
    
    print()
    print("=" * 80)
    print("🎉 COMPLETE!")
    print("=" * 80)
    
    return original_text, completed_text


# -------------------------------
# 5. EXAMPLE USAGE
# -------------------------------
if __name__ == "__main__":
    # Example: Process a handwriting image
    image_path = "/kaggle/input/test-iam/test_image.png"
    
    # Run the pipeline
    original, completed = autocomplete_handwriting(
        image_path=image_path,
        num_words=3  # Add only 3 words
    )
    
    # You can also use the functions separately:
    
    # Just OCR:
    # text = ocr_image("your_image.png")
    
    # Just text completion:
    # completed = complete_text("The writing's on the wall", num_words=3)

🖥️ Device: cuda

📖 Loading TrOCR (OCR model)...


Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-large-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🤖 Loading GPT-2 (Language model)...
✅ Models loaded!

🚀 HANDWRITING AUTO-COMPLETE

📋 STEP 1: Optical Character Recognition (OCR)
--------------------------------------------------------------------------------
🔍 Reading image: /kaggle/input/test-iam/test_image.png
📝 Extracted text: 'The writing's on the wall'

📋 STEP 2: Text Completion
--------------------------------------------------------------------------------
🤖 Generating 3-word continuation...
✨ Completed text: 'The writing's on the wall I suppose'

📋 STEP 3: Results
--------------------------------------------------------------------------------
📝 Original:  The writing's on the wall
➕ Added:     I suppose
✅ Completed: The writing's on the wall I suppose

🎉 COMPLETE!
